In [ ]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1) Paths to the 3 separate IMDb aggregate result files
# ---------------------------------------------------------
sf025_csv = Path("/path/to/schemalens/imdb/results/sf_025/benchmark_aggregate_results.csv")
sf05_csv  = Path("/path/to/schemalens/imdb/results/sf_050/benchmark_aggregate_results.csv")
sf1_csv   = Path("/path/to/schemalens/imdb/results/sf_1/benchmark_aggregate_results.csv")

# ---------------------------------------------------------
# 2) Read each file
# ---------------------------------------------------------
df025 = pd.read_csv(sf025_csv)
df05  = pd.read_csv(sf05_csv)
df1   = pd.read_csv(sf1_csv)

# ---------------------------------------------------------
# 3) Ensure scale_label is correct
#    (only overwrite if needed)
# ---------------------------------------------------------
df025["scale_label"] = "sf0.25"
df05["scale_label"]  = "sf0.5"
df1["scale_label"]   = "sf1"

# ---------------------------------------------------------
# 4) Concatenate
# ---------------------------------------------------------
all_df = pd.concat([df025, df05, df1], ignore_index=True)

# ---------------------------------------------------------
# 5) Optional sanity checks
# ---------------------------------------------------------
print("Rows:", len(all_df))
print("Scale labels:", sorted(all_df["scale_label"].dropna().unique()))
print("Run phases:", sorted(all_df["run_phase"].dropna().unique()))
print("Queries:", sorted(all_df["query_name"].dropna().unique()))

# ---------------------------------------------------------
# 6) Save consolidated file
# ---------------------------------------------------------
out_csv = Path("benchmark_aggregate_results_imdb_all_sfs.csv")
all_df.to_csv(out_csv, index=False)

print(f"\nSaved: {out_csv.resolve()}")

In [ ]:

import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path

# =========================================================
# IMDb RESULTS ANALYSIS — USING REAL EXPERIMENT CATALOG
# =========================================================
# Inputs expected in the same folder:
# - benchmark_aggregate_results.csv
# - mongo_experiment_catalog(1).csv
#
# This notebook:
# 1) reads the real experiment catalog
# 2) derives the activated family for each IMDb query from the catalog
# 3) computes the metrics per (query, scale factor, run phase)
# 4) exports the corrected artifacts
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
results_csv = Path("/path/to/schemalens/imdb/results/benchmark_aggregate_results_imdb_all_sfs.csv")
catalog_csv = Path("/path/to/schemalens/imdb/mongo_experiment_catalog.csv")
run_phase_to_analyze = "hot"
near_best_threshold = 0.05

output_dir = Path("imdb_analysis_outputs_catalog_corrected")
output_dir.mkdir(parents=True, exist_ok=True)

# Representative IMDb cases used later in the paper
representative_queries = {
    "association": "QG3_RecommendationByGenreAndSubtype",
    "associative": "QG4_AllPersonsOfTypeForWatchItem",
    "containment": "QG6_EpisodesOfSeries",
}

# ---------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------
def parse_query_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    # Try Python-list syntax first
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
    # Fallback: comma-separated
    return [x.strip() for x in text.split(",") if x.strip()]

def official_id_from_query(query_name: str):
    m = re.match(r"^(QG\d+)", str(query_name))
    return m.group(1) if m else str(query_name)

# ---------------------------------------------------------
# 3) Read files
# ---------------------------------------------------------
agg = pd.read_csv(results_csv)
catalog = pd.read_csv(catalog_csv)

required_agg_cols = [
    "activated_class",
    "benchmark_family",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "p95_latency_ms",
]
missing_agg = [c for c in required_agg_cols if c not in agg.columns]
if missing_agg:
    raise ValueError(f"Missing required columns in benchmark_aggregate_results.csv: {missing_agg}")

required_cat_cols = [
    "activated_class",
    "primary_queries",
]
missing_cat = [c for c in required_cat_cols if c not in catalog.columns]
if missing_cat:
    raise ValueError(f"Missing required columns in mongo_experiment_catalog(1).csv: {missing_cat}")

agg["official_id"] = agg["query_name"].apply(official_id_from_query)

# Restrict to one run phase
phase_df = agg[agg["run_phase"] == run_phase_to_analyze].copy()
if phase_df.empty:
    raise ValueError(f"No rows found for run_phase = {run_phase_to_analyze!r}")

# ---------------------------------------------------------
# 4) Derive real activated family from the experiment catalog
# ---------------------------------------------------------
query_to_activated = {}

for _, row in catalog.iterrows():
    activated_class = str(row["activated_class"]).strip()
    primary_queries = parse_query_list(row.get("primary_queries"))
    for q in primary_queries:
        query_to_activated.setdefault(q, set()).add(activated_class)

query_to_activated = {k: sorted(v) for k, v in query_to_activated.items()}

# Global broader space size derived from the actually benchmarked classes
full_space_classes = sorted(phase_df["activated_class"].dropna().astype(str).unique())
n_C = len(full_space_classes)

print("Broader tested configuration space:", full_space_classes)
print("Size of C:", n_C)

activated_family_df = pd.DataFrame([
    {
        "query_name": q,
        "official_id": official_id_from_query(q),
        "activated_family": ", ".join(v),
        "n_activated_configs_from_catalog": len(v),
    }
    for q, v in sorted(query_to_activated.items())
])

print("\nActivated family derived from the experiment catalog:")
display(activated_family_df)

# ---------------------------------------------------------
# 5) Compute corrected metrics per (query, scale, run_phase)
# ---------------------------------------------------------
rows = []

group_cols = ["query_name", "scale_label", "run_phase"]

for (query_name, scale_label, run_phase), grp in phase_df.groupby(group_cols):
    grp = grp.copy()

    # Full-space best
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_latency_full = float(best_all["p95_latency_ms"])
    best_config_full = str(best_all["activated_class"])

    # Real activated family from catalog
    activated_family = query_to_activated.get(query_name, [])
    activated_family_set = set(activated_family)

    # Activated subset present in the benchmark results for this query/scale/phase
    activated_grp = grp[grp["activated_class"].astype(str).isin(activated_family_set)].copy()

    n_tested_configs = grp["activated_class"].nunique()
    n_activated_configs = len(activated_family_set)
    dsr = 1 - (n_activated_configs / n_C) if n_C > 0 else np.nan

    # Top-1 preservation = the best config in the full space belongs to activated family
    top1_preserved_by_activated = best_config_full in activated_family_set

    # Near-best preservation (within 5% of the best full-space latency)
    if best_latency_full > 0:
        near_best_mask = ((grp["p95_latency_ms"] - best_latency_full) / best_latency_full) <= near_best_threshold
        near_best_classes = set(grp.loc[near_best_mask, "activated_class"].dropna().astype(str).unique())
        near_best_preserved_by_activated = len(activated_family_set.intersection(near_best_classes)) > 0
    else:
        near_best_preserved_by_activated = False

    # Activated regret
    if activated_grp.empty:
        activated_regret = np.nan
        best_activated_config = None
        best_activated_p95 = np.nan
    else:
        best_activated = activated_grp.loc[activated_grp["p95_latency_ms"].idxmin()]
        best_activated_config = best_activated["activated_class"]
        best_activated_p95 = float(best_activated["p95_latency_ms"])
        activated_regret = (best_activated_p95 - best_latency_full) / best_latency_full if best_latency_full > 0 else np.nan

    # Best primary config in the actual benchmark results
    primary = grp[grp["query_group"] == "primary"].copy()
    if primary.empty:
        best_primary_config = None
        best_primary_p95_ms = np.nan
        primary_regret = np.nan
    else:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        best_primary_config = best_primary["activated_class"]
        best_primary_p95_ms = float(best_primary["p95_latency_ms"])
        primary_regret = (best_primary_p95_ms - best_latency_full) / best_latency_full if best_latency_full > 0 else np.nan

    rows.append({
        "official_id": official_id_from_query(query_name),
        "query_name": query_name,
        "scale_label": scale_label,
        "run_phase": run_phase,

        "n_tested_configs": int(n_tested_configs),
        "n_activated_configs": int(n_activated_configs),
        "DSR": float(dsr) if pd.notna(dsr) else np.nan,

        "activated_family": ", ".join(activated_family),
        "best_config": best_all["activated_class"],
        "best_group": best_all["query_group"],
        "best_design_pattern": best_all["benchmark_family"],
        "best_p95_ms": best_latency_full,

        "top1_preserved_by_activated": bool(top1_preserved_by_activated),
        "near_best_preserved_by_activated": bool(near_best_preserved_by_activated),
        "activated_regret": float(activated_regret) if pd.notna(activated_regret) else np.nan,
        "best_activated_config": best_activated_config,
        "best_activated_p95_ms": best_activated_p95,

        "best_primary_config": best_primary_config,
        "best_primary_p95_ms": best_primary_p95_ms,
        "primary_regret": float(primary_regret) if pd.notna(primary_regret) else np.nan,
    })

analysis_df = pd.DataFrame(rows).sort_values(
    by=["official_id", "scale_label", "run_phase"]
).reset_index(drop=True)

print("\nCorrected IMDb query-level summary using the real catalog:\n")
display(analysis_df)

# ---------------------------------------------------------
# 6) Global metrics
# ---------------------------------------------------------
print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Near-best preservation activated:", analysis_df["near_best_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].dropna().mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())

# ---------------------------------------------------------
# 7) Representative IMDb cases for the paper
# ---------------------------------------------------------
rep_rows = []
for family_name, query_name in representative_queries.items():
    subset = analysis_df[analysis_df["query_name"] == query_name].copy()
    if subset.empty:
        continue

    # choose the row with the smallest activated regret across scales
    chosen = subset.loc[subset["activated_regret"].fillna(np.inf).idxmin()]

    rep_rows.append({
        "dataset": "IMDb",
        "family": family_name.capitalize(),
        "representative_query": query_name,
        "scale_label": chosen["scale_label"],
        "activated_configs": chosen["activated_family"],
        "best_config": chosen["best_config"],
        "DSR": chosen["DSR"],
        "Top-1": chosen["top1_preserved_by_activated"],
        "Near-best(5%)": chosen["near_best_preserved_by_activated"],
        "Regret": chosen["activated_regret"],
    })

representative_df = pd.DataFrame(rep_rows)
print("\nRepresentative IMDb cases for the paper:")
display(representative_df)

# ---------------------------------------------------------
# 8) Secondary/control winners tables
# ---------------------------------------------------------
secondary_winners_df = analysis_df[
    analysis_df["best_group"] == "secondary_affected"
][[
    "official_id",
    "query_name",
    "scale_label",
    "activated_family",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]].reset_index(drop=True)

control_winners_df = analysis_df[
    analysis_df["best_group"] == "control"
][[
    "official_id",
    "query_name",
    "scale_label",
    "activated_family",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]].reset_index(drop=True)

print("\nSecondary-affected winners:")
display(secondary_winners_df)

print("\nControl winners:")
display(control_winners_df)

# ---------------------------------------------------------
# 9) Best-by-query-scale artifact
# ---------------------------------------------------------
best_by_query_scale_rows = []

for (query_name, scale_label), grp in phase_df.groupby(["query_name", "scale_label"]):
    best = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_by_query_scale_rows.append({
        "official_id": official_id_from_query(query_name),
        "query_name": query_name,
        "scale_label": scale_label,
        "best_config": best["activated_class"],
        "best_group": best["query_group"],
        "best_design_pattern": best["benchmark_family"],
        "best_p95_ms": best["p95_latency_ms"],
        "best_avg_latency_ms": best.get("avg_latency_ms", np.nan),
        "best_p99_ms": best.get("p99_latency_ms", np.nan),
        "avg_documents_returned": best.get("avg_documents_returned", np.nan),
    })

best_by_query_scale_df = pd.DataFrame(best_by_query_scale_rows).sort_values(
    by=["official_id", "scale_label"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 10) Export artifacts
# ---------------------------------------------------------
analysis_df.to_csv(output_dir / f"imdb_summary_{run_phase_to_analyze}_catalog_corrected.csv", index=False)
representative_df.to_csv(output_dir / f"imdb_representative_cases_{run_phase_to_analyze}.csv", index=False)
secondary_winners_df.to_csv(output_dir / f"imdb_secondary_winners_{run_phase_to_analyze}_catalog_corrected.csv", index=False)
control_winners_df.to_csv(output_dir / f"imdb_control_winners_{run_phase_to_analyze}_catalog_corrected.csv", index=False)
best_by_query_scale_df.to_csv(output_dir / f"imdb_best_by_query_scale_{run_phase_to_analyze}_catalog_corrected.csv", index=False)

print(f"\nSaved outputs in: {output_dir.resolve()}")
